
# LV benchmark patch: anchored eSSVI → Dupire → backward PDE

This notebook **replaces Section 11 onward** of the original LV notebook. Run Sections 1–10 of the original notebook first, then run this patch from top to bottom.

The patch addresses the earlier implementation problems by:

1. recomputing bid, midpoint and ask implied volatilities with one Black–Scholes convention;
2. replacing the restrictive global Heston-like SSVI fit with sequential **anchored eSSVI slices**;
3. calibrating in option-price units rather than total-variance units;
4. separating surface-calibration error from Dupire/PDE consistency error;
5. using one forward curve and one discount curve per calibration date;
6. evaluating frozen next-day local volatility at elapsed **model time**, rather than restarting the coefficient at time zero;
7. using versioned caches and exactly one expensive calculation per cache block.

The main references are:

- Gatheral and Jacquier, *Arbitrage-Free SVI Volatility Surfaces*, especially Sections 4–5: [arXiv:1204.0646](https://arxiv.org/abs/1204.0646).
- Corbetta, Cohort, Laachir and Martini, *Robust Calibration and Arbitrage-Free Interpolation of SSVI Slices*: [arXiv:1804.04924](https://arxiv.org/abs/1804.04924).
- Gatheral, *The Volatility Surface*, Chapter 1, for Dupire local volatility.

The implementation is deliberately minimal. It does not add neural smoothing, Andreasen–Huge interpolation, Monte Carlo pricing or automatic parameter regularisation.



## 11. Consistent implied-volatility targets and deterministic market curves

For each quote, recompute

$$
\sigma_{\rm bid},\qquad \sigma_{\rm mid},\qquad \sigma_{\rm ask}
$$

using exactly the same forward price, discount factor, maturity and Black–Scholes inversion. The fitted total variance is

$$
w_{\rm mid}=\sigma_{\rm mid}^2T.
$$

This removes the former inconsistency in which midpoint implied volatility could come from a vendor convention while bid and ask implied volatilities came from the notebook convention.

A **forward curve** on one date means the function

$$
T\longmapsto F(0,T),
$$

containing the forward prices for all expiries on that date. A **discount curve** means

$$
T\longmapsto D(0,T).
$$

The notebook linearly interpolates $\log F(0,T)$ and $\log D(0,T)$ between observed maturity nodes. Hence the instantaneous carry and interest rate are piecewise constant:

$$
b(T)=\frac{\partial}{\partial T}\log F(0,T),
\qquad
r(T)=-\frac{\partial}{\partial T}\log D(0,T).
$$

Using one pair of curves per date ensures that every expiry is priced under one common deterministic rate-and-dividend specification.


In [ ]:

from scipy.optimize import minimize_scalar

PATCH_VERSION = "anchored_essvi_v1"

# Corbetta et al. report that a coarse rho grid is normally sufficient.
# We use 21 points, then refine twice around the best point.
ESSVI_RHO_GRID_POINTS = 21
ESSVI_RHO_REFINEMENTS = 3
ESSVI_RHO_LIMIT = 0.999
ESSVI_PARAMETER_EPS = 1e-10
MAX_ANCHOR_ABS_K = 0.05

# Only dates passing this explicit same-day surface-fit gate reach Dupire.
MIN_DATE_INSIDE_BID_ASK = 0.90
MAX_DATE_MEDIAN_ERROR_TO_HALF_SPREAD = 1.00

# The PDE must reproduce the fitted eSSVI surface more accurately than the
# market spread before the date is used for next-day testing.
MAX_PDE_MEDIAN_ERROR_TO_HALF_SPREAD = 0.25

PATCH_CACHE_FOLDER = CACHE_FOLDER / PATCH_VERSION
PATCH_CACHE_FOLDER.mkdir(parents=True, exist_ok=True)

calibration_cache = PATCH_CACHE_FOLDER / "calibration_quotes.pkl"
curve_nodes_cache = PATCH_CACHE_FOLDER / "market_curve_nodes.pkl"


def compute_consistent_implied_volatilities(data):
    bid_values = []
    mid_values = []
    ask_values = []

    for row in tqdm(
        data.itertuples(index=False),
        total=len(data),
        desc="Computing bid, midpoint and ask implied volatilities",
    ):
        common_arguments = (
            row.cp_flag,
            row.forward_price,
            row.strike,
            row.T,
            row.discount,
        )

        bid_values.append(
            implied_volatility_from_price(
                *common_arguments,
                row.best_bid,
            )
        )
        mid_values.append(
            implied_volatility_from_price(
                *common_arguments,
                row.mid,
            )
        )
        ask_values.append(
            implied_volatility_from_price(
                *common_arguments,
                row.best_offer,
            )
        )

    result = data.copy()
    result["bid_iv"] = bid_values
    result["market_iv"] = mid_values
    result["ask_iv"] = ask_values

    return result


if calibration_cache.exists():
    print("Loading cached, consistently inverted calibration quotes.")
    calibration_quotes = pd.read_pickle(calibration_cache)

else:
    calibration_quotes = compute_consistent_implied_volatilities(
        calibration_quotes
    )

    valid_iv = (
        calibration_quotes["bid_iv"].between(0.001, 5.0)
        & calibration_quotes["market_iv"].between(0.001, 5.0)
        & calibration_quotes["ask_iv"].between(0.001, 5.0)
        & calibration_quotes["bid_iv"].le(
            calibration_quotes["market_iv"]
        )
        & calibration_quotes["market_iv"].le(
            calibration_quotes["ask_iv"]
        )
    )

    calibration_quotes = calibration_quotes.loc[valid_iv].copy()

    calibration_quotes["bid_total_variance"] = (
        calibration_quotes["bid_iv"] ** 2
        * calibration_quotes["T"]
    )
    calibration_quotes["market_total_variance"] = (
        calibration_quotes["market_iv"] ** 2
        * calibration_quotes["T"]
    )
    calibration_quotes["ask_total_variance"] = (
        calibration_quotes["ask_iv"] ** 2
        * calibration_quotes["T"]
    )

    calibration_quotes["half_spread"] = np.maximum(
        0.5 * (
            calibration_quotes["best_offer"]
            - calibration_quotes["best_bid"]
        ),
        1e-8,
    )

    calibration_quotes.to_pickle(
        calibration_cache
    )
    print("Saved consistent calibration quotes.")

print(f"Calibration quotes available: {len(calibration_quotes):,}")


def build_market_curve_nodes(data):
    rows = []

    for date, day_data in data.groupby("date"):
        spot_value = float(day_data["spot"].median())

        maturity_rows = (
            day_data.groupby("T", as_index=False)
            .agg(
                forward_price=("forward_price", "median"),
                discount=("discount", "median"),
            )
            .sort_values("T")
        )

        rows.append(
            {
                "date": pd.Timestamp(date),
                "T": 0.0,
                "spot": spot_value,
                "forward_price": spot_value,
                "discount": 1.0,
            }
        )

        for row in maturity_rows.itertuples(index=False):
            rows.append(
                {
                    "date": pd.Timestamp(date),
                    "T": float(row.T),
                    "spot": spot_value,
                    "forward_price": float(row.forward_price),
                    "discount": float(row.discount),
                }
            )

    return pd.DataFrame(rows)


if curve_nodes_cache.exists():
    print("Loading cached market-curve nodes.")
    market_curve_nodes = pd.read_pickle(curve_nodes_cache)

else:
    market_curve_nodes = build_market_curve_nodes(quotes_all)
    market_curve_nodes.to_pickle(curve_nodes_cache)
    print("Saved market-curve nodes.")


def rebuild_market_curves(curve_nodes):
    curves = {}

    for date, day_nodes in curve_nodes.groupby("date"):
        day_nodes = day_nodes.sort_values("T")

        times = day_nodes["T"].to_numpy(dtype=float)
        forwards = day_nodes["forward_price"].to_numpy(dtype=float)
        discounts = day_nodes["discount"].to_numpy(dtype=float)

        if (
            len(times) < 2
            or np.any(np.diff(times) <= 0)
            or np.any(forwards <= 0)
            or np.any(discounts <= 0)
        ):
            continue

        curves[pd.Timestamp(date)] = {
            "times": times,
            "log_forwards": np.log(forwards),
            "log_discounts": np.log(discounts),
            "spot": float(day_nodes["spot"].iloc[0]),
        }

    return curves


market_curves = rebuild_market_curves(market_curve_nodes)


def piecewise_linear_value(time, nodes, values):
    time_array = np.asarray(time, dtype=float)
    clipped_time = np.clip(time_array, nodes[0], nodes[-1])
    return np.interp(clipped_time, nodes, values)


def piecewise_linear_slope(time, nodes, values):
    time_array = np.asarray(time, dtype=float)
    clipped_time = np.clip(time_array, nodes[0], nodes[-1])

    interval = np.searchsorted(
        nodes,
        clipped_time,
        side="right",
    ) - 1
    interval = np.clip(interval, 0, len(nodes) - 2)

    return (
        values[interval + 1] - values[interval]
    ) / (
        nodes[interval + 1] - nodes[interval]
    )


def forward_from_curve(curve, time):
    return np.exp(
        piecewise_linear_value(
            time,
            curve["times"],
            curve["log_forwards"],
        )
    )


def discount_from_curve(curve, time):
    return np.exp(
        piecewise_linear_value(
            time,
            curve["times"],
            curve["log_discounts"],
        )
    )


def instantaneous_carry(curve, time):
    return piecewise_linear_slope(
        time,
        curve["times"],
        curve["log_forwards"],
    )


def instantaneous_rate(curve, time):
    return -piecewise_linear_slope(
        time,
        curve["times"],
        curve["log_discounts"],
    )


def discount_between(curve, time_left, time_right):
    return (
        discount_from_curve(curve, time_right)
        / discount_from_curve(curve, time_left)
    )


def dividend_discount_between(curve, time_left, time_right):
    return (
        discount_between(curve, time_left, time_right)
        * forward_from_curve(curve, time_right)
        / forward_from_curve(curve, time_left)
    )

print(f"Market curves available: {len(market_curves):,}")



## 12. Sequential anchored eSSVI calibration in option-price units

For one maturity, write

$$
\psi=\theta\varphi,
$$

so the eSSVI slice is

$$
w(k)
=
\frac12\left[
\theta+\rho\psi k
+
\sqrt{\theta^2+2\rho\psi\theta k+\psi^2k^2}
\right].
$$

Corbetta et al. choose the reliable market point $(k^*,\theta^*)$ closest to forward ATM and use the anchored reparameterisation

$$
\theta=\theta^*-\rho\psi k^*.
$$

Their paper explicitly describes this as the first-order solution of $\theta^*=w(k^*,\theta)$. We use their stated parameterisation rather than introducing an additional calibrated $\theta$.

For one slice, the Gatheral–Jacquier sufficient butterfly conditions become

$$
\psi<\frac{4}{1+|\rho|},
\qquad
\psi<2\sqrt{\frac{\theta}{1+|\rho|}}.
$$

The second inequality yields the explicit upper bound $\psi_+(\rho,k^*,\theta^*)$ given in Corbetta et al., Section 2.1.

For consecutive slices $(\theta_-,\rho_-,\psi_-)$ and $(\theta,\rho,\psi)$, absence of calendar-spread arbitrage is guaranteed by

$$
\theta\geq\theta_-,
\qquad
\psi\geq\psi_-,
\qquad
|\rho\psi-\rho_-\psi_-|\leq\psi-\psi_-.
$$

For each sampled $\rho$, these conditions define an interval of feasible $\psi$. A one-dimensional Brent minimisation then minimises the mean absolute option-price error:

$$
\frac1n\sum_{j=1}^n
\left|
C_j^{\rm eSSVI}-C_j^{\rm mid}
\right|.
$$

This follows the calibration structure proposed in Corbetta et al.: sample $\rho$, optimise $\psi$ in one dimension, select the best pair, and proceed from short to long maturity.


In [ ]:

def essvi_total_variance(k, theta, rho, psi):
    k = np.asarray(k, dtype=float)

    radicand = (
        theta**2
        + 2.0 * rho * psi * theta * k
        + psi**2 * k**2
    )

    return 0.5 * (
        theta
        + rho * psi * k
        + np.sqrt(np.maximum(radicand, 0.0))
    )


def anchored_theta(theta_star, k_star, rho, psi):
    return theta_star - rho * psi * k_star


def butterfly_psi_upper(rho, k_star, theta_star):
    denominator = 1.0 + abs(rho)

    return (
        -2.0 * rho * k_star / denominator
        + np.sqrt(
            4.0 * rho**2 * k_star**2 / denominator**2
            + 4.0 * theta_star / denominator
        )
    )


def feasible_psi_interval(rho, anchor, previous_slice=None):
    k_star = anchor["k_star"]
    theta_star = anchor["theta_star"]

    lower = ESSVI_PARAMETER_EPS
    upper = min(
        4.0 / (1.0 + abs(rho)),
        butterfly_psi_upper(rho, k_star, theta_star),
    )

    rho_times_k = rho * k_star

    # theta = theta_star - rho * psi * k_star must remain positive.
    if rho_times_k > 0:
        upper = min(
            upper,
            theta_star / rho_times_k,
        )

    if previous_slice is not None:
        previous_theta = previous_slice["theta"]
        previous_psi = previous_slice["psi"]
        previous_chi = previous_slice["rho"] * previous_psi

        # Equivalent form of
        # |rho*psi - previous_rho*previous_psi|
        # <= psi - previous_psi.
        calendar_lower = max(
            (previous_psi - previous_chi) / (1.0 - rho),
            (previous_psi + previous_chi) / (1.0 + rho),
        )

        lower = max(lower, previous_psi, calendar_lower)

        # Enforce theta >= previous_theta.
        if abs(rho_times_k) < 1e-14:
            if theta_star < previous_theta:
                return None

        elif rho_times_k > 0:
            upper = min(
                upper,
                (theta_star - previous_theta) / rho_times_k,
            )

        else:
            lower = max(
                lower,
                (theta_star - previous_theta) / rho_times_k,
            )

    lower += ESSVI_PARAMETER_EPS
    upper -= ESSVI_PARAMETER_EPS

    if not np.isfinite(lower) or not np.isfinite(upper):
        return None

    if upper <= lower:
        return None

    return float(lower), float(upper)


def essvi_price_objective(
    psi,
    rho,
    slice_data,
    anchor,
):
    theta = anchored_theta(
        anchor["theta_star"],
        anchor["k_star"],
        rho,
        psi,
    )

    if theta <= 0:
        return np.inf

    total_variance = essvi_total_variance(
        slice_data["log_moneyness"].to_numpy(dtype=float),
        theta,
        rho,
        psi,
    )

    if np.any(total_variance <= 0):
        return np.inf

    fitted_iv = np.sqrt(
        total_variance / float(slice_data["T"].iloc[0])
    )

    fitted_price = bs_price(
        slice_data["cp_flag"].to_numpy(),
        slice_data["forward_price"].to_numpy(dtype=float),
        slice_data["strike"].to_numpy(dtype=float),
        slice_data["T"].to_numpy(dtype=float),
        slice_data["discount"].to_numpy(dtype=float),
        fitted_iv,
    )

    return float(
        np.mean(
            np.abs(
                fitted_price
                - slice_data["mid"].to_numpy(dtype=float)
            )
        )
    )


def fit_one_essvi_slice(slice_data, previous_slice=None):
    slice_data = slice_data.sort_values("log_moneyness").copy()

    anchor_index = (
        slice_data["log_moneyness"].abs().idxmin()
    )
    anchor_row = slice_data.loc[anchor_index]

    anchor = {
        "k_star": float(anchor_row["log_moneyness"]),
        "theta_star": float(anchor_row["market_total_variance"]),
    }

    if abs(anchor["k_star"]) > MAX_ANCHOR_ABS_K:
        return None, "no sufficiently near-ATM anchor"

    rho_left = -ESSVI_RHO_LIMIT
    rho_right = ESSVI_RHO_LIMIT
    best = None

    for _ in range(ESSVI_RHO_REFINEMENTS):
        rho_grid = np.linspace(
            rho_left,
            rho_right,
            ESSVI_RHO_GRID_POINTS,
        )

        refinement_results = []

        for rho in rho_grid:
            interval = feasible_psi_interval(
                float(rho),
                anchor,
                previous_slice,
            )

            if interval is None:
                continue

            lower, upper = interval

            result = minimize_scalar(
                essvi_price_objective,
                bounds=(lower, upper),
                method="bounded",
                args=(float(rho), slice_data, anchor),
                options={"xatol": 1e-10},
            )

            if result.success and np.isfinite(result.fun):
                refinement_results.append(
                    {
                        "rho": float(rho),
                        "psi": float(result.x),
                        "objective": float(result.fun),
                    }
                )

        if not refinement_results:
            return None, "empty feasible eSSVI parameter set"

        current_best = min(
            refinement_results,
            key=lambda row: row["objective"],
        )

        if best is None or current_best["objective"] < best["objective"]:
            best = current_best

        grid_step = (
            rho_right - rho_left
        ) / (ESSVI_RHO_GRID_POINTS - 1)

        rho_left = max(
            -ESSVI_RHO_LIMIT,
            best["rho"] - grid_step,
        )
        rho_right = min(
            ESSVI_RHO_LIMIT,
            best["rho"] + grid_step,
        )

    theta = anchored_theta(
        anchor["theta_star"],
        anchor["k_star"],
        best["rho"],
        best["psi"],
    )

    fitted_slice = {
        "theta": float(theta),
        "rho": float(best["rho"]),
        "psi": float(best["psi"]),
        "chi": float(best["rho"] * best["psi"]),
        "phi": float(best["psi"] / theta),
        "objective_mae": float(best["objective"]),
        "k_star": anchor["k_star"],
        "theta_star": anchor["theta_star"],
    }

    return fitted_slice, "success"


def fit_one_essvi_date(day_data):
    quote_counts = day_data.groupby("T").size()
    eligible_maturities = quote_counts.loc[
        quote_counts.ge(MIN_QUOTES_PER_EXPIRY)
    ].index

    day_data = day_data.loc[
        day_data["T"].isin(eligible_maturities)
    ].copy()

    rows = []
    failure_rows = []
    previous_slice = None

    for maturity, slice_data in day_data.groupby("T", sort=True):
        fitted_slice, message = fit_one_essvi_slice(
            slice_data,
            previous_slice,
        )

        if fitted_slice is None:
            failure_rows.append(
                {
                    "T": float(maturity),
                    "status": message,
                }
            )
            continue

        row = {
            "T": float(maturity),
            **fitted_slice,
            "number_of_quotes": len(slice_data),
            "status": message,
        }
        rows.append(row)
        previous_slice = row

    if len(rows) < MIN_EXPIRIES_PER_DATE:
        return None, failure_rows

    return pd.DataFrame(rows), failure_rows


def rebuild_essvi_models(parameter_table):
    models = {}

    for date, day_parameters in parameter_table.groupby("date"):
        day_parameters = day_parameters.sort_values("T")

        models[pd.Timestamp(date)] = {
            "maturities": day_parameters["T"].to_numpy(dtype=float),
            "theta": day_parameters["theta"].to_numpy(dtype=float),
            "psi": day_parameters["psi"].to_numpy(dtype=float),
            "chi": day_parameters["chi"].to_numpy(dtype=float),
            "rho": day_parameters["rho"].to_numpy(dtype=float),
        }

    return models


parameter_cache = PATCH_CACHE_FOLDER / "essvi_parameters.pkl"
failure_cache = PATCH_CACHE_FOLDER / "essvi_failures.pkl"

if parameter_cache.exists():
    print("Loading cached anchored-eSSVI parameters.")
    essvi_parameters = pd.read_pickle(parameter_cache)

    if failure_cache.exists():
        essvi_failures = pd.read_pickle(failure_cache)
    else:
        essvi_failures = pd.DataFrame()

else:
    parameter_frames = []
    failure_frames = []

    for date, day_data in tqdm(
        calibration_quotes.groupby("date", sort=True),
        total=calibration_quotes["date"].nunique(),
        desc="Calibrating anchored eSSVI date by date",
    ):
        fitted_parameters, failures = fit_one_essvi_date(day_data)

        if fitted_parameters is not None:
            fitted_parameters["date"] = pd.Timestamp(date)
            parameter_frames.append(fitted_parameters)

        if failures:
            failure_frame = pd.DataFrame(failures)
            failure_frame["date"] = pd.Timestamp(date)
            failure_frames.append(failure_frame)

    essvi_parameters = (
        pd.concat(parameter_frames, ignore_index=True)
        if parameter_frames
        else pd.DataFrame()
    )
    essvi_failures = (
        pd.concat(failure_frames, ignore_index=True)
        if failure_frames
        else pd.DataFrame()
    )

    essvi_parameters.to_pickle(parameter_cache)
    essvi_failures.to_pickle(failure_cache)
    print("Saved anchored-eSSVI parameters and failures.")

essvi_models = rebuild_essvi_models(essvi_parameters)

print(f"Dates with fitted eSSVI models: {len(essvi_models):,}")
print(f"Fitted maturity slices: {len(essvi_parameters):,}")

if not essvi_parameters.empty:
    display(
        essvi_parameters[
            [
                "date",
                "T",
                "theta",
                "psi",
                "rho",
                "objective_mae",
                "k_star",
            ]
        ].head(12)
    )



## 13. Direct same-day surface fit and acceptance gate

The calibration stage is evaluated **before** Dupire or the PDE.

For each calibration quote, the fitted surface price is

$$
C^{\rm surf}
=
C_{\rm BS}\!\left(
F,K,T,D,
\sqrt{\frac{w^{\rm eSSVI}(k,T)}{T}}
\right).
$$

This isolates the surface-calibration error

$$
C^{\rm surf}-C^{\rm mid}.
$$

Only dates satisfying the declared gate continue:

- at least 90% of OTM calibration prices lie inside bid–ask;
- the median absolute price error is no more than one half-spread.

This does not claim that eSSVI can fit every dataset. It guarantees that all dates admitted to the LV benchmark have already demonstrated a good same-day surface fit under the stated criterion.


In [ ]:

def evaluate_fitted_slice_quotes(slice_data, parameter_row):
    total_variance = essvi_total_variance(
        slice_data["log_moneyness"].to_numpy(dtype=float),
        float(parameter_row["theta"]),
        float(parameter_row["rho"]),
        float(parameter_row["psi"]),
    )

    fitted_iv = np.sqrt(
        total_variance / slice_data["T"].to_numpy(dtype=float)
    )

    fitted_price = bs_price(
        slice_data["cp_flag"].to_numpy(),
        slice_data["forward_price"].to_numpy(dtype=float),
        slice_data["strike"].to_numpy(dtype=float),
        slice_data["T"].to_numpy(dtype=float),
        slice_data["discount"].to_numpy(dtype=float),
        fitted_iv,
    )

    output = slice_data.copy()
    output["essvi_total_variance"] = total_variance
    output["essvi_iv"] = fitted_iv
    output["essvi_surface_price"] = fitted_price
    output["surface_price_error"] = (
        output["essvi_surface_price"] - output["mid"]
    )
    output["surface_inside_bid_ask"] = (
        output["essvi_surface_price"].between(
            output["best_bid"],
            output["best_offer"],
        )
    )
    output["surface_error_to_half_spread"] = (
        np.abs(output["surface_price_error"])
        / output["half_spread"]
    )

    return output


surface_fit_cache = PATCH_CACHE_FOLDER / "surface_fit_quotes.pkl"
date_fit_cache = PATCH_CACHE_FOLDER / "surface_fit_by_date.pkl"

if surface_fit_cache.exists() and date_fit_cache.exists():
    print("Loading cached direct eSSVI surface-fit diagnostics.")
    surface_fit_quotes = pd.read_pickle(surface_fit_cache)
    surface_fit_by_date = pd.read_pickle(date_fit_cache)

else:
    evaluated_frames = []

    parameter_lookup = {
        (pd.Timestamp(row.date), float(row.T)): row._asdict()
        for row in essvi_parameters.itertuples(index=False)
    }

    for (date, maturity), slice_data in tqdm(
        calibration_quotes.groupby(["date", "T"], sort=True),
        total=calibration_quotes.groupby(["date", "T"]).ngroups,
        desc="Evaluating direct eSSVI surface prices",
    ):
        parameter_row = parameter_lookup.get(
            (pd.Timestamp(date), float(maturity))
        )

        if parameter_row is None:
            continue

        evaluated_frames.append(
            evaluate_fitted_slice_quotes(
                slice_data,
                parameter_row,
            )
        )

    surface_fit_quotes = (
        pd.concat(evaluated_frames, ignore_index=True)
        if evaluated_frames
        else pd.DataFrame()
    )

    date_rows = []

    for date, day_data in surface_fit_quotes.groupby("date"):
        date_rows.append(
            {
                "date": pd.Timestamp(date),
                "number_of_quotes": len(day_data),
                "number_of_expiries": day_data["T"].nunique(),
                "price_rmse": float(
                    np.sqrt(
                        np.mean(day_data["surface_price_error"] ** 2)
                    )
                ),
                "price_mae": float(
                    np.mean(np.abs(day_data["surface_price_error"]))
                ),
                "inside_bid_ask": float(
                    day_data["surface_inside_bid_ask"].mean()
                ),
                "median_error_to_half_spread": float(
                    day_data["surface_error_to_half_spread"].median()
                ),
                "p95_error_to_half_spread": float(
                    day_data["surface_error_to_half_spread"].quantile(0.95)
                ),
            }
        )

    surface_fit_by_date = pd.DataFrame(date_rows)
    surface_fit_by_date["accepted_surface_fit"] = (
        surface_fit_by_date["inside_bid_ask"].ge(
            MIN_DATE_INSIDE_BID_ASK
        )
        & surface_fit_by_date["median_error_to_half_spread"].le(
            MAX_DATE_MEDIAN_ERROR_TO_HALF_SPREAD
        )
    )

    surface_fit_quotes.to_pickle(surface_fit_cache)
    surface_fit_by_date.to_pickle(date_fit_cache)
    print("Saved direct eSSVI surface-fit diagnostics.")

accepted_surface_dates = set(
    pd.to_datetime(
        surface_fit_by_date.loc[
            surface_fit_by_date["accepted_surface_fit"],
            "date",
        ]
    )
)

print(
    f"Dates passing the same-day surface-fit gate: "
    f"{len(accepted_surface_dates)} / {len(surface_fit_by_date)}"
)
display(surface_fit_by_date.sort_values("date"))


# Verify the no-arbitrage slice inequalities directly from the fitted nodes.
def adjacent_slice_checks(parameter_table):
    rows = []

    for date, day_parameters in parameter_table.groupby("date"):
        day_parameters = day_parameters.sort_values("T")

        previous = None

        for row in day_parameters.itertuples(index=False):
            butterfly_one = (
                row.psi * (1.0 + abs(row.rho))
            )
            butterfly_two = (
                row.psi**2 * (1.0 + abs(row.rho))
                / row.theta
            )

            result = {
                "date": pd.Timestamp(date),
                "T": float(row.T),
                "butterfly_condition_one": butterfly_one < 4.0,
                "butterfly_condition_two": butterfly_two <= 4.0,
                "theta_non_decreasing": True,
                "psi_non_decreasing": True,
                "calendar_slope_condition": True,
            }

            if previous is not None:
                result["theta_non_decreasing"] = (
                    row.theta >= previous.theta - 1e-12
                )
                result["psi_non_decreasing"] = (
                    row.psi >= previous.psi - 1e-12
                )
                result["calendar_slope_condition"] = (
                    abs(row.chi - previous.chi)
                    <= row.psi - previous.psi + 1e-12
                )

            rows.append(result)
            previous = row

    return pd.DataFrame(rows)


slice_constraint_checks = adjacent_slice_checks(essvi_parameters)

print("Failed fitted-node constraints:")
display(
    slice_constraint_checks.loc[
        ~slice_constraint_checks[
            [
                "butterfly_condition_one",
                "butterfly_condition_two",
                "theta_non_decreasing",
                "psi_non_decreasing",
                "calendar_slope_condition",
            ]
        ].all(axis=1)
    ]
)


# Plot six representative expiries for one accepted date.
if accepted_surface_dates:
    example_date = sorted(accepted_surface_dates)[0]
    example_data = surface_fit_quotes.loc[
        surface_fit_quotes["date"].eq(example_date)
    ].copy()

    maturities = np.sort(example_data["T"].unique())
    selected_indices = np.linspace(
        0,
        len(maturities) - 1,
        min(6, len(maturities)),
        dtype=int,
    )
    selected_maturities = maturities[selected_indices]

    figure, axes = plt.subplots(
        len(selected_maturities),
        1,
        figsize=(8, 2.8 * len(selected_maturities)),
        sharex=False,
    )

    axes = np.atleast_1d(axes)

    for axis, maturity in zip(axes, selected_maturities):
        maturity_data = example_data.loc[
            example_data["T"].eq(maturity)
        ].sort_values("log_moneyness")

        axis.scatter(
            maturity_data["log_moneyness"],
            maturity_data["bid_iv"],
            s=12,
            label="Bid IV",
        )
        axis.scatter(
            maturity_data["log_moneyness"],
            maturity_data["ask_iv"],
            s=12,
            label="Ask IV",
        )
        axis.plot(
            maturity_data["log_moneyness"],
            maturity_data["essvi_iv"],
            linewidth=1.5,
            label="Anchored eSSVI",
        )
        axis.set_title(f"T = {maturity:.4f}")
        axis.set_ylabel("Implied volatility")
        axis.grid(alpha=0.25)

    axes[-1].set_xlabel("Forward log-moneyness k")
    axes[0].legend()
    figure.tight_layout()
    plt.show()



## 14. Arbitrage-free maturity interpolation and Dupire local volatility

For consecutive fitted slices, define

$$
\chi_i=\rho_i\psi_i.
$$

Corbetta et al., Section 5, interpolate linearly in

$$
\theta,\qquad \psi,\qquad \chi=\rho\psi.
$$

For $T_i\leq T\leq T_{i+1}$, with

$$
\lambda=\frac{T-T_i}{T_{i+1}-T_i},
$$

set

$$
\theta(T)=(1-\lambda)\theta_i+\lambda\theta_{i+1},
$$

$$
\psi(T)=(1-\lambda)\psi_i+\lambda\psi_{i+1},
$$

$$
\chi(T)=(1-\lambda)\chi_i+\lambda\chi_{i+1},
\qquad
\rho(T)=\frac{\chi(T)}{\psi(T)}.
$$

Before the first maturity, use their short-end extrapolation

$$
\theta(T)=\frac{T}{T_1}\theta_1,
\quad
\psi(T)=\frac{T}{T_1}\psi_1,
\quad
\rho(T)=\rho_1.
$$

In the variables $(\theta,\psi,\chi)$, eSSVI becomes

$$
w(k,T)
=
\frac12\left[
\theta+\chi k+
\sqrt{\theta^2+2\chi\theta k+\psi^2k^2}
\right].
$$

The notebook differentiates this expression analytically in $k$ and in $T$. Dupire local variance is

$$
\sigma_{\rm loc}^2(k,T)
=
\frac{w_T}{g},
$$

where

$$
g=
\left(1-\frac{k w_k}{2w}\right)^2
-
\frac{w_k^2}{4}\left(\frac1w+\frac14\right)
+
\frac12w_{kk}.
$$

No local-volatility floor or cap is applied. A date with materially negative $w_T$, non-positive $g$, or negative local variance is rejected rather than silently repaired.


In [ ]:

def essvi_parameters_at_time(model, time):
    time = float(time)
    maturities = model["maturities"]
    theta_nodes = model["theta"]
    psi_nodes = model["psi"]
    chi_nodes = model["chi"]

    first_maturity = maturities[0]
    last_maturity = maturities[-1]

    if time <= first_maturity:
        time_used = max(time, 1e-12)
        scale = time_used / first_maturity

        theta = scale * theta_nodes[0]
        psi = scale * psi_nodes[0]
        chi = scale * chi_nodes[0]

        theta_T = theta_nodes[0] / first_maturity
        psi_T = psi_nodes[0] / first_maturity
        chi_T = chi_nodes[0] / first_maturity

    else:
        if time > last_maturity + 1e-10:
            raise ValueError(
                "Requested model time exceeds the last fitted maturity."
            )

        time_used = min(time, last_maturity)

        interval = np.searchsorted(
            maturities,
            time_used,
            side="right",
        ) - 1
        interval = min(interval, len(maturities) - 2)

        time_left = maturities[interval]
        time_right = maturities[interval + 1]
        weight = (
            time_used - time_left
        ) / (
            time_right - time_left
        )

        theta = (
            (1.0 - weight) * theta_nodes[interval]
            + weight * theta_nodes[interval + 1]
        )
        psi = (
            (1.0 - weight) * psi_nodes[interval]
            + weight * psi_nodes[interval + 1]
        )
        chi = (
            (1.0 - weight) * chi_nodes[interval]
            + weight * chi_nodes[interval + 1]
        )

        theta_T = (
            theta_nodes[interval + 1] - theta_nodes[interval]
        ) / (
            time_right - time_left
        )
        psi_T = (
            psi_nodes[interval + 1] - psi_nodes[interval]
        ) / (
            time_right - time_left
        )
        chi_T = (
            chi_nodes[interval + 1] - chi_nodes[interval]
        ) / (
            time_right - time_left
        )

    return {
        "theta": float(theta),
        "psi": float(psi),
        "chi": float(chi),
        "rho": float(chi / psi),
        "theta_T": float(theta_T),
        "psi_T": float(psi_T),
        "chi_T": float(chi_T),
    }


def essvi_surface_terms(k, parameters):
    k = np.asarray(k, dtype=float)

    theta = parameters["theta"]
    psi = parameters["psi"]
    chi = parameters["chi"]
    theta_T = parameters["theta_T"]
    psi_T = parameters["psi_T"]
    chi_T = parameters["chi_T"]

    radicand = (
        theta**2
        + 2.0 * chi * theta * k
        + psi**2 * k**2
    )
    square_root = np.sqrt(np.maximum(radicand, 1e-24))

    w = 0.5 * (
        theta + chi * k + square_root
    )

    numerator_k = (
        chi * theta + psi**2 * k
    )

    w_k = 0.5 * (
        chi + numerator_k / square_root
    )

    w_kk = (
        0.5
        * theta**2
        * (psi**2 - chi**2)
        / square_root**3
    )

    radicand_T_half = (
        theta * theta_T
        + k * (chi_T * theta + chi * theta_T)
        + psi * psi_T * k**2
    )

    w_T = 0.5 * (
        theta_T
        + chi_T * k
        + radicand_T_half / square_root
    )

    dupire_denominator = (
        (1.0 - k * w_k / (2.0 * w)) ** 2
        - 0.25 * w_k**2 * (1.0 / w + 0.25)
        + 0.5 * w_kk
    )

    local_variance = w_T / dupire_denominator

    return {
        "w": w,
        "w_k": w_k,
        "w_kk": w_kk,
        "w_T": w_T,
        "dupire_denominator": dupire_denominator,
        "local_variance": local_variance,
    }


def essvi_terms_from_model(model, k, time):
    parameters = essvi_parameters_at_time(model, time)
    return essvi_surface_terms(k, parameters)


def essvi_local_volatility(model, k, time):
    terms = essvi_terms_from_model(model, k, time)

    w_T = np.asarray(terms["w_T"], dtype=float)
    denominator = np.asarray(
        terms["dupire_denominator"],
        dtype=float,
    )
    local_variance = np.asarray(
        terms["local_variance"],
        dtype=float,
    )

    valid = (
        w_T >= -1e-10
    ) & (
        denominator > 0.0
    ) & (
        local_variance >= -1e-10
    )

    local_variance = np.where(
        valid,
        np.maximum(local_variance, 0.0),
        np.nan,
    )

    return np.sqrt(local_variance)


def diagnose_one_essvi_model(model):
    maturity_grid = np.linspace(
        max(1.0 / 365.0, model["maturities"][0] / 5.0),
        model["maturities"][-1],
        40,
    )
    k_grid = np.linspace(
        -DIAGNOSTIC_K_MAX,
        DIAGNOSTIC_K_MAX,
        121,
    )

    w_T_values = []
    denominator_values = []
    local_variance_values = []

    for maturity in maturity_grid:
        terms = essvi_terms_from_model(
            model,
            k_grid,
            maturity,
        )

        w_T_values.extend(np.ravel(terms["w_T"]))
        denominator_values.extend(
            np.ravel(terms["dupire_denominator"])
        )
        local_variance_values.extend(
            np.ravel(terms["local_variance"])
        )

    w_T_values = np.asarray(w_T_values)
    denominator_values = np.asarray(denominator_values)
    local_variance_values = np.asarray(local_variance_values)

    return {
        "minimum_w_T": float(np.nanmin(w_T_values)),
        "fraction_w_T_negative": float(
            np.mean(w_T_values < -1e-10)
        ),
        "minimum_dupire_denominator": float(
            np.nanmin(denominator_values)
        ),
        "fraction_dupire_denominator_nonpositive": float(
            np.mean(denominator_values <= 0.0)
        ),
        "minimum_local_variance": float(
            np.nanmin(local_variance_values)
        ),
        "fraction_local_variance_negative": float(
            np.mean(local_variance_values < -1e-10)
        ),
    }


validity_cache = PATCH_CACHE_FOLDER / "dupire_validity_by_date.pkl"

if validity_cache.exists():
    print("Loading cached Dupire-validity diagnostics.")
    dupire_validity_by_date = pd.read_pickle(validity_cache)

else:
    validity_rows = []

    for date in tqdm(
        sorted(accepted_surface_dates),
        desc="Checking eSSVI interpolation and Dupire validity",
    ):
        model = essvi_models.get(pd.Timestamp(date))

        if model is None:
            continue

        row = diagnose_one_essvi_model(model)
        row["date"] = pd.Timestamp(date)
        validity_rows.append(row)

    dupire_validity_by_date = pd.DataFrame(validity_rows)
    dupire_validity_by_date["accepted_dupire"] = (
        dupire_validity_by_date["fraction_w_T_negative"].eq(0.0)
        & dupire_validity_by_date[
            "fraction_dupire_denominator_nonpositive"
        ].eq(0.0)
        & dupire_validity_by_date[
            "fraction_local_variance_negative"
        ].eq(0.0)
    )

    dupire_validity_by_date.to_pickle(validity_cache)
    print("Saved Dupire-validity diagnostics.")

accepted_dupire_dates = set(
    pd.to_datetime(
        dupire_validity_by_date.loc[
            dupire_validity_by_date["accepted_dupire"],
            "date",
        ]
    )
)

print(
    f"Dates passing both surface fit and Dupire validity: "
    f"{len(accepted_dupire_dates)}"
)
display(dupire_validity_by_date.sort_values("date"))



## 15. Backward local-volatility PDE, Crank–Nicolson and Rannacher start-up

Under the deterministic local-volatility model

$$
dS_t=b(t)S_t\,dt+\sigma_{\rm loc}(t,S_t)S_t\,dW_t,
$$

an option value $V(t,S)$ satisfies

$$
V_t
+
\frac12\sigma_{\rm loc}^2(t,S)S^2V_{SS}
+
b(t)SV_S
-
r(t)V
=0.
$$

Using $x=\log(S/S_0)$, this becomes

$$
V_t
+
\frac12\sigma_{\rm loc}^2V_{xx}
+
\left(b-\frac12\sigma_{\rm loc}^2\right)V_x
-
rV
=0.
$$

Crank–Nicolson averages the spatial PDE operator between two adjacent time levels. It is second-order accurate in time for a smooth solution, but the European payoff has a **kink** at $S=K$: its first derivative jumps. Rannacher start-up replaces the first two Crank–Nicolson intervals adjacent to the payoff by four implicit-Euler half-steps. Implicit Euler damps the alternating grid-scale numerical error generated by the kink, after which Crank–Nicolson is used normally.

For frozen next-day pricing, `model_time_offset` is the elapsed time since calibration. If one day has elapsed, the coefficient used at solver time $u$ is

$$
\sigma_{\rm loc}^{(t)}(\Delta t+u,S),
$$

not $\sigma_{\rm loc}^{(t)}(u,S)$. This is the precise purpose of the offset.


In [ ]:

def option_boundaries(
    strikes,
    cp_flags,
    S_min,
    S_max,
    absolute_time,
    absolute_expiry,
    curve,
):
    rate_discount = discount_between(
        curve,
        absolute_time,
        absolute_expiry,
    )
    dividend_discount = dividend_discount_between(
        curve,
        absolute_time,
        absolute_expiry,
    )

    lower_boundary = np.zeros(len(strikes))
    upper_boundary = np.zeros(len(strikes))

    calls = cp_flags == "C"
    puts = cp_flags == "P"

    upper_boundary[calls] = np.maximum(
        S_max * dividend_discount
        - strikes[calls] * rate_discount,
        0.0,
    )

    lower_boundary[puts] = np.maximum(
        strikes[puts] * rate_discount
        - S_min * dividend_discount,
        0.0,
    )

    return lower_boundary, upper_boundary


def make_essvi_local_volatility_function(
    model,
    curve,
    model_time_offset,
):
    def local_volatility_function(time_from_valuation, state_spot):
        model_time = model_time_offset + time_from_valuation
        forward_at_model_time = forward_from_curve(
            curve,
            model_time,
        )

        state_log_moneyness = np.log(
            np.asarray(state_spot, dtype=float)
            / forward_at_model_time
        )

        return essvi_local_volatility(
            model,
            state_log_moneyness,
            model_time,
        )

    return local_volatility_function


def one_backward_time_step(
    value_right,
    time_right,
    time_left,
    model_time_offset,
    absolute_expiry,
    x_grid,
    spot_grid,
    strikes,
    cp_flags,
    curve,
    local_volatility_function,
    implicit_weight,
):
    time_step = time_right - time_left
    time_middle = 0.5 * (time_right + time_left)
    absolute_time_middle = model_time_offset + time_middle

    local_volatility = np.asarray(
        local_volatility_function(
            time_middle,
            spot_grid,
        ),
        dtype=float,
    )

    if not np.all(np.isfinite(local_volatility)):
        raise ValueError(
            "The local-volatility surface is invalid on the PDE grid."
        )

    rate = float(instantaneous_rate(curve, absolute_time_middle))
    carry = float(instantaneous_carry(curve, absolute_time_middle))

    dx = x_grid[1] - x_grid[0]

    diffusion = 0.5 * local_volatility**2
    drift = carry - 0.5 * local_volatility**2

    lower_operator = diffusion / dx**2 - drift / (2.0 * dx)
    diagonal_operator = -2.0 * diffusion / dx**2 - rate
    upper_operator = diffusion / dx**2 + drift / (2.0 * dx)

    inside = slice(1, len(x_grid) - 1)

    lower_left = (
        -implicit_weight
        * time_step
        * lower_operator[inside]
    )
    diagonal_left = (
        1.0
        - implicit_weight
        * time_step
        * diagonal_operator[inside]
    )
    upper_left = (
        -implicit_weight
        * time_step
        * upper_operator[inside]
    )

    explicit_weight = 1.0 - implicit_weight

    right_hand_side = (
        (
            1.0
            + explicit_weight
            * time_step
            * diagonal_operator[inside]
        )[:, None]
        * value_right[1:-1]
        + (
            explicit_weight
            * time_step
            * lower_operator[inside]
        )[:, None]
        * value_right[:-2]
        + (
            explicit_weight
            * time_step
            * upper_operator[inside]
        )[:, None]
        * value_right[2:]
    )

    absolute_time_left = model_time_offset + time_left

    lower_boundary, upper_boundary = option_boundaries(
        strikes,
        cp_flags,
        spot_grid[0],
        spot_grid[-1],
        absolute_time_left,
        absolute_expiry,
        curve,
    )

    right_hand_side[0] -= lower_left[0] * lower_boundary
    right_hand_side[-1] -= upper_left[-1] * upper_boundary

    number_of_interior_points = len(x_grid) - 2
    banded_matrix = np.zeros((3, number_of_interior_points))
    banded_matrix[0, 1:] = upper_left[:-1]
    banded_matrix[1] = diagonal_left
    banded_matrix[2, :-1] = lower_left[1:]

    value_left = value_right.copy()
    value_left[0] = lower_boundary
    value_left[-1] = upper_boundary
    value_left[1:-1] = solve_banded(
        (1, 1),
        banded_matrix,
        right_hand_side,
    )

    return value_left


def solve_local_volatility_pde_group(
    spot,
    model,
    curve,
    model_time_offset,
    absolute_expiry,
    strikes,
    cp_flags,
    local_volatility_function=None,
    space_points=PDE_SPACE_POINTS,
    time_steps_per_year=PDE_TIME_STEPS_PER_YEAR,
    log_spot_width=PDE_LOG_SPOT_WIDTH,
    rannacher_full_steps=RANNACHER_FULL_STEPS,
):
    strikes = np.asarray(strikes, dtype=float)
    cp_flags = np.asarray(cp_flags)

    remaining_maturity = absolute_expiry - model_time_offset

    if remaining_maturity <= 0:
        price = np.where(
            cp_flags == "C",
            np.maximum(spot - strikes, 0.0),
            np.maximum(strikes - spot, 0.0),
        )
        delta = np.where(
            cp_flags == "C",
            (spot > strikes).astype(float),
            -(spot < strikes).astype(float),
        )
        return price, delta

    if local_volatility_function is None:
        local_volatility_function = (
            make_essvi_local_volatility_function(
                model,
                curve,
                model_time_offset,
            )
        )

    if space_points % 2 == 0:
        space_points += 1

    time_steps = max(
        40,
        int(np.ceil(time_steps_per_year * remaining_maturity)),
    )
    time_steps = max(time_steps, rannacher_full_steps + 1)

    x_grid = np.linspace(
        -log_spot_width,
        log_spot_width,
        space_points,
    )
    spot_grid = spot * np.exp(x_grid)
    dx = x_grid[1] - x_grid[0]

    value = np.where(
        cp_flags[None, :] == "C",
        np.maximum(
            spot_grid[:, None] - strikes[None, :],
            0.0,
        ),
        np.maximum(
            strikes[None, :] - spot_grid[:, None],
            0.0,
        ),
    )

    full_time_step = remaining_maturity / time_steps
    time_right = remaining_maturity

    for step_number in range(time_steps):
        if step_number < rannacher_full_steps:
            for _ in range(2):
                time_left = time_right - 0.5 * full_time_step

                value = one_backward_time_step(
                    value,
                    time_right,
                    time_left,
                    model_time_offset,
                    absolute_expiry,
                    x_grid,
                    spot_grid,
                    strikes,
                    cp_flags,
                    curve,
                    local_volatility_function,
                    implicit_weight=1.0,
                )

                time_right = time_left

        else:
            time_left = time_right - full_time_step

            value = one_backward_time_step(
                value,
                time_right,
                time_left,
                model_time_offset,
                absolute_expiry,
                x_grid,
                spot_grid,
                strikes,
                cp_flags,
                curve,
                local_volatility_function,
                implicit_weight=0.5,
            )

            time_right = time_left

    centre = space_points // 2
    price = value[centre].copy()

    value_x = (
        value[centre + 1] - value[centre - 1]
    ) / (2.0 * dx)
    delta = value_x / spot

    return price, delta


# Constant-volatility validation of the PDE machinery.
def make_flat_market_curve(spot, rate, dividend_yield, maximum_time):
    times = np.array([0.0, maximum_time], dtype=float)
    carry = rate - dividend_yield

    return {
        "times": times,
        "log_forwards": np.log(spot) + carry * times,
        "log_discounts": -rate * times,
        "spot": float(spot),
    }


def constant_local_volatility_function(sigma):
    def function(time_from_valuation, state_spot):
        return np.full_like(
            np.asarray(state_spot, dtype=float),
            sigma,
            dtype=float,
        )

    return function


validation_spot = 100.0
validation_rate = 0.03
validation_dividend = 0.01
validation_sigma = 0.20
validation_maturity = 0.50
validation_strikes = np.array([80.0, 100.0, 120.0])
validation_flags = np.array(["C", "C", "C"])

validation_curve = make_flat_market_curve(
    validation_spot,
    validation_rate,
    validation_dividend,
    validation_maturity,
)

pde_price, pde_delta = solve_local_volatility_pde_group(
    validation_spot,
    model=None,
    curve=validation_curve,
    model_time_offset=0.0,
    absolute_expiry=validation_maturity,
    strikes=validation_strikes,
    cp_flags=validation_flags,
    local_volatility_function=(
        constant_local_volatility_function(validation_sigma)
    ),
)

validation_forward = forward_from_curve(
    validation_curve,
    validation_maturity,
)
validation_discount = discount_from_curve(
    validation_curve,
    validation_maturity,
)

bs_validation_price = bs_price(
    validation_flags,
    validation_forward,
    validation_strikes,
    validation_maturity,
    validation_discount,
    validation_sigma,
)
bs_validation_delta = bs_delta(
    validation_flags,
    validation_spot,
    validation_forward,
    validation_strikes,
    validation_maturity,
    validation_discount,
    validation_sigma,
)

constant_volatility_validation = pd.DataFrame(
    {
        "strike": validation_strikes,
        "pde_price": pde_price,
        "bs_price": bs_validation_price,
        "price_error": pde_price - bs_validation_price,
        "pde_delta": pde_delta,
        "bs_delta": bs_validation_delta,
        "delta_error": pde_delta - bs_validation_delta,
    }
)

display(constant_volatility_validation)



## 16. Same-day PDE recovery and market benchmark

For every admitted date, calculate three prices:

1. $C^{\rm mid}$: observed market midpoint;
2. $C^{\rm surf}$: Black–Scholes price generated directly from the fitted eSSVI total variance;
3. $C^{\rm LV}$: backward-PDE price generated from the Dupire local volatility.

The two distinct errors are

$$
C^{\rm surf}-C^{\rm mid}
\quad\text{and}\quad
C^{\rm LV}-C^{\rm surf}.
$$

The first measures calibration quality. The second measures whether the Dupire conversion, interpolation and PDE reproduce the fitted European surface. The next-day benchmark uses only dates whose median PDE-recovery error is at most one quarter of a market half-spread.


In [ ]:

def fitted_surface_price_from_model(data, models):
    output = pd.Series(np.nan, index=data.index, dtype=float)

    for (date, maturity), group in data.groupby(["date", "T"]):
        model = models.get(pd.Timestamp(date))

        if model is None:
            continue

        try:
            parameters = essvi_parameters_at_time(
                model,
                float(maturity),
            )
        except ValueError:
            continue

        terms = essvi_surface_terms(
            group["log_moneyness"].to_numpy(dtype=float),
            parameters,
        )

        total_variance = terms["w"]
        fitted_iv = np.sqrt(
            total_variance / group["T"].to_numpy(dtype=float)
        )

        output.loc[group.index] = bs_price(
            group["cp_flag"].to_numpy(),
            group["forward_price"].to_numpy(dtype=float),
            group["strike"].to_numpy(dtype=float),
            group["T"].to_numpy(dtype=float),
            group["discount"].to_numpy(dtype=float),
            fitted_iv,
        )

    return output


def price_dataframe_by_groups(
    data,
    models,
    curves,
    model_date_column,
    current_date_column,
    spot_column,
    absolute_expiry_column,
    strike_column,
    cp_flag_column,
    progress_description,
):
    output = pd.DataFrame(
        index=data.index,
        columns=["model_price", "model_delta"],
        dtype=float,
    )

    group_columns = []

    for column in [
        model_date_column,
        current_date_column,
        "exdate",
        "am_settlement",
    ]:
        if column not in group_columns:
            group_columns.append(column)

    grouped_data = data.groupby(
        group_columns,
        dropna=False,
        sort=True,
    )

    for _, group in tqdm(
        grouped_data,
        total=len(grouped_data),
        desc=progress_description,
    ):
        model_date = pd.Timestamp(
            group[model_date_column].iloc[0]
        )
        current_date = pd.Timestamp(
            group[current_date_column].iloc[0]
        )

        model = models.get(model_date)
        curve = curves.get(model_date)

        if model is None or curve is None:
            continue

        model_time_offset = (
            current_date - model_date
        ).days / 365.0

        absolute_expiry = float(
            group[absolute_expiry_column].iloc[0]
        )

        if absolute_expiry > model["maturities"][-1] + 1e-10:
            continue

        spot_value = float(group[spot_column].iloc[0])

        try:
            model_price, model_delta = (
                solve_local_volatility_pde_group(
                    spot=spot_value,
                    model=model,
                    curve=curve,
                    model_time_offset=model_time_offset,
                    absolute_expiry=absolute_expiry,
                    strikes=group[strike_column].to_numpy(dtype=float),
                    cp_flags=group[cp_flag_column].to_numpy(),
                )
            )
        except (ValueError, np.linalg.LinAlgError):
            continue

        output.loc[group.index, "model_price"] = model_price
        output.loc[group.index, "model_delta"] = model_delta

    return output


same_day_cache = PATCH_CACHE_FOLDER / "same_day_pde_result.pkl"

same_day_input = quotes_all.loc[
    quotes_all["date"].isin(accepted_dupire_dates)
].copy()

if same_day_cache.exists():
    print("Loading cached same-day PDE results.")
    same_day_result = pd.read_pickle(same_day_cache)

else:
    same_day_result = price_dataframe_by_groups(
        same_day_input,
        essvi_models,
        market_curves,
        model_date_column="date",
        current_date_column="date",
        spot_column="spot",
        absolute_expiry_column="T",
        strike_column="strike",
        cp_flag_column="cp_flag",
        progress_description="Same-day eSSVI-LV PDE pricing",
    )
    same_day_result.to_pickle(same_day_cache)
    print("Saved same-day PDE results.")

lv_quotes = same_day_input.copy()
lv_quotes["essvi_surface_price"] = fitted_surface_price_from_model(
    lv_quotes,
    essvi_models,
)
lv_quotes["lv_price"] = same_day_result["model_price"]
lv_quotes["lv_delta"] = same_day_result["model_delta"]

lv_quotes = lv_quotes.dropna(
    subset=["essvi_surface_price", "lv_price", "lv_delta"]
).copy()

lv_quotes["surface_market_error"] = (
    lv_quotes["essvi_surface_price"] - lv_quotes["mid"]
)
lv_quotes["pde_surface_recovery_error"] = (
    lv_quotes["lv_price"] - lv_quotes["essvi_surface_price"]
)
lv_quotes["lv_market_error"] = (
    lv_quotes["lv_price"] - lv_quotes["mid"]
)
lv_quotes["half_spread"] = np.maximum(
    0.5 * (lv_quotes["best_offer"] - lv_quotes["best_bid"]),
    1e-8,
)
lv_quotes["pde_recovery_error_to_half_spread"] = (
    np.abs(lv_quotes["pde_surface_recovery_error"])
    / lv_quotes["half_spread"]
)
lv_quotes["lv_inside_bid_ask"] = lv_quotes["lv_price"].between(
    lv_quotes["best_bid"],
    lv_quotes["best_offer"],
)

pde_recovery_rows = []

for date, day_data in lv_quotes.groupby("date"):
    pde_recovery_rows.append(
        {
            "date": pd.Timestamp(date),
            "number_of_quotes": len(day_data),
            "surface_market_mae": float(
                np.mean(np.abs(day_data["surface_market_error"]))
            ),
            "pde_surface_recovery_mae": float(
                np.mean(
                    np.abs(day_data["pde_surface_recovery_error"])
                )
            ),
            "lv_market_mae": float(
                np.mean(np.abs(day_data["lv_market_error"]))
            ),
            "lv_inside_bid_ask": float(
                day_data["lv_inside_bid_ask"].mean()
            ),
            "median_pde_error_to_half_spread": float(
                day_data[
                    "pde_recovery_error_to_half_spread"
                ].median()
            ),
        }
    )

pde_recovery_by_date = pd.DataFrame(pde_recovery_rows)
pde_recovery_by_date["accepted_pde_recovery"] = (
    pde_recovery_by_date[
        "median_pde_error_to_half_spread"
    ].le(MAX_PDE_MEDIAN_ERROR_TO_HALF_SPREAD)
)

accepted_pde_dates = set(
    pd.to_datetime(
        pde_recovery_by_date.loc[
            pde_recovery_by_date["accepted_pde_recovery"],
            "date",
        ]
    )
)

same_day_summary = pd.Series(
    {
        "number_of_quotes": len(lv_quotes),
        "surface_market_rmse": float(
            np.sqrt(np.mean(lv_quotes["surface_market_error"] ** 2))
        ),
        "surface_market_mae": float(
            np.mean(np.abs(lv_quotes["surface_market_error"]))
        ),
        "pde_surface_recovery_rmse": float(
            np.sqrt(
                np.mean(lv_quotes["pde_surface_recovery_error"] ** 2)
            )
        ),
        "pde_surface_recovery_mae": float(
            np.mean(
                np.abs(lv_quotes["pde_surface_recovery_error"])
            )
        ),
        "lv_market_rmse": float(
            np.sqrt(np.mean(lv_quotes["lv_market_error"] ** 2))
        ),
        "lv_market_mae": float(
            np.mean(np.abs(lv_quotes["lv_market_error"]))
        ),
        "lv_inside_bid_ask": float(
            lv_quotes["lv_inside_bid_ask"].mean()
        ),
    }
)

print(
    f"Dates passing the PDE surface-recovery gate: "
    f"{len(accepted_pde_dates)}"
)
display(same_day_summary)
display(pde_recovery_by_date.sort_values("date"))



## 17. Frozen next-day pricing and one-day delta hedge

Suppose the model is calibrated on date $t$, and the next observation is $t+\Delta t$.

The frozen model retains:

- the day-$t$ eSSVI parameters;
- the day-$t$ forward curve;
- the day-$t$ discount curve.

At next-day solver time $u$, the local-volatility coefficient is evaluated at absolute model time

$$
\Delta t+u.
$$

The same original absolute expiry $T$ is retained, so the remaining maturity is

$$
T-\Delta t.
$$

This fixes the former clock-restart error. The hedge uses the day-$t$ PDE delta and the same one-day hedging-error definition as the BS-IVF notebook:

$$
\varepsilon_{t,t+1}
=
(C_{t+1}-C_t)
-
\Delta_t(S_{t+1}-S_t).
$$

Interest earned on the cash component and dividend cashflows are omitted here only to preserve direct comparability with the existing BS-IVF benchmark.


In [ ]:

next_day_base = evaluation_pairs.merge(
    lv_quotes[
        [
            "date",
            "optionid",
            "lv_price",
            "lv_delta",
        ]
    ],
    on=["date", "optionid"],
    how="inner",
    validate="one_to_one",
)

next_day_base = next_day_base.loc[
    next_day_base["date"].isin(accepted_pde_dates)
].copy()

next_day_cache = PATCH_CACHE_FOLDER / "next_day_pde_result.pkl"

if next_day_cache.exists():
    print("Loading cached frozen next-day PDE results.")
    next_day_result = pd.read_pickle(next_day_cache)

else:
    next_day_result = price_dataframe_by_groups(
        next_day_base,
        essvi_models,
        market_curves,
        model_date_column="date",
        current_date_column="date_next",
        spot_column="spot_next",
        absolute_expiry_column="T",
        strike_column="strike_next",
        cp_flag_column="cp_flag_next",
        progress_description="Frozen next-day eSSVI-LV PDE pricing",
    )
    next_day_result.to_pickle(next_day_cache)
    print("Saved frozen next-day PDE results.")

lv_pairs = next_day_base.copy()
lv_pairs["next_day_model_price"] = next_day_result["model_price"]

lv_pairs = lv_pairs.dropna(
    subset=["next_day_model_price", "lv_delta"]
).copy()

lv_pairs["next_day_price_error"] = (
    lv_pairs["next_day_model_price"] - lv_pairs["mid_next"]
)
lv_pairs["hedging_error"] = (
    lv_pairs["option_change"]
    - lv_pairs["lv_delta"]
    * (lv_pairs["spot_next"] - lv_pairs["spot"])
)

lv_summary = pd.DataFrame(
    {
        "all_clean_options": summarise_model(
            lv_pairs,
            "next_day_price_error",
            "next_day_model_price",
            "hedging_error",
        ),
        "otm_options": summarise_model(
            lv_pairs.loc[lv_pairs["is_otm"]],
            "next_day_price_error",
            "next_day_model_price",
            "hedging_error",
        ),
    }
).T

display(lv_summary)



## 18. Save outputs and interpretation order

Interpret the results in this order:

1. `surface_fit_by_date`: did anchored eSSVI fit market prices well before Dupire?
2. `dupire_validity_by_date`: did interpolation produce non-negative $w_T$ and positive Dupire denominator?
3. `pde_recovery_by_date`: did the PDE recover the fitted eSSVI surface?
4. `same_day_summary`: how close was the accepted LV model to the market on the calibration date?
5. `lv_summary`: how did the frozen model price and hedge one trading day later?

A missing date is a reported model failure, not an invitation to replace invalid local volatility silently by a floor or cap.


In [ ]:

output_folder = Path("local_vol_essvi_patch_outputs")
output_folder.mkdir(exist_ok=True)

essvi_parameters.to_csv(
    output_folder / "essvi_slice_parameters.csv",
    index=False,
)
essvi_failures.to_csv(
    output_folder / "essvi_slice_failures.csv",
    index=False,
)
surface_fit_by_date.to_csv(
    output_folder / "surface_fit_by_date.csv",
    index=False,
)
dupire_validity_by_date.to_csv(
    output_folder / "dupire_validity_by_date.csv",
    index=False,
)
pde_recovery_by_date.to_csv(
    output_folder / "pde_recovery_by_date.csv",
    index=False,
)
lv_quotes.to_csv(
    output_folder / "same_day_quote_results.csv",
    index=False,
)
lv_pairs.to_csv(
    output_folder / "next_day_and_hedging_results.csv",
    index=False,
)
same_day_summary.to_csv(
    output_folder / "same_day_summary.csv",
    header=["value"],
)
lv_summary.to_csv(
    output_folder / "next_day_and_hedging_summary.csv"
)

print("Saved patch outputs to:", output_folder.resolve())



## References

1. J. Gatheral and A. Jacquier (2014), “Arbitrage-Free SVI Volatility Surfaces,” *Quantitative Finance* 14(1), 59–71. Preprint dated March 2013: [arXiv:1204.0646](https://arxiv.org/abs/1204.0646).
2. J. Corbetta, P. Cohort, I. Laachir and C. Martini (2019), “Robust Calibration and Arbitrage-Free Interpolation of SSVI Slices,” *Decisions in Economics and Finance* 42, 665–677: [arXiv:1804.04924](https://arxiv.org/abs/1804.04924).
3. J. Gatheral (2006), *The Volatility Surface: A Practitioner’s Guide*, Chapter 1.
4. R. Rannacher (1984), “Finite Element Solution of Diffusion Problems with Irregular Data,” *Numerische Mathematik* 43, 309–327. The notebook uses the standard option-PDE application: implicit-Euler half-steps followed by Crank–Nicolson.
